In [ ]:
from config import MODEL_CONFIGS
from pathlib import Path
from testing import load_dataset, load_tokenizer, setup_paths
from config import MODEL_CONFIGS, TEST_TYPES, MAX_GEN_TOKEN_LENGTH
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch
from peft import PeftModel

In [ ]:
# Set model configuration
model_name = "qwen_coder_1p5b"
test_type = "fine_tuned_code"

if model_name not in MODEL_CONFIGS:
    raise ValueError(
        f"Unknown model '{model_name}'. "
        f"Available models: {list(MODEL_CONFIGS.keys())}"
    )

if test_type not in TEST_TYPES:
    raise ValueError(
        f"Unknown test type '{test_type}'. "
        f"Available types: {TEST_TYPES}"
    )

# Setup paths
paths = setup_paths(model_name, test_type)

In [ ]:
def load_model(model_name: str, adapter_dir: Path, test_type: str):
    """Load base model and optionally apply LoRA adapter."""

    print(f"Loading model: {model_name}")

    bnb_config = BitsAndBytesConfig( # for testing on local machine
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",          
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
        )
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",               
        dtype=torch.float16,
    )
    
    if test_type in ["fine_tuned_code", "fine_tuned_patch"]:
        print(f"Loading LoRA adapter from: {adapter_dir}")
        model = PeftModel.from_pretrained(
            model,
            str(adapter_dir),
            dtype=torch.bfloat16,
        )
    
    model.eval()
    return model

model = load_model(model_name, paths['adapter_dir'], test_type)
tokenizer = load_tokenizer(model_name)
ds_test = load_dataset(paths["test_data"], test_type, tokenizer)

In [ ]:
def ask_model(prompt):

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_GEN_TOKEN_LENGTH
        )

    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[:, input_length:]
    
    answer = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
        )

    return answer

In [52]:
# Debugging by testing a example
i = 6
example = ds_test[i]
print(example["prompt"])
answer = ask_model(example["messages"])[0]
print(answer)

Check the SysML v2 code below for correctness with respect to the given rules.
If the code is incorrect, produce a corrected version by modifying the existing code.
If the code already satisfies all rules, it should simply be reported as correct.

Domain Rules:
- 'DriveIF' belongs to Domain: mechanical_torque
- 'ShaftPort_d' belongs to Domain: mechanical_torque
- 'HandPort' belongs to Domain: command_signal

Valid Connections Rules:
- command_signal can ONLY connect to: ['command_signal']
- mechanical_torque can ONLY connect to: ['mechanical_torque']

### Code: 
```sysml
package Vehicle_Remix_0939_418a {
    port def DriveIF;
    port def HandPort;
    port def ShaftPort_d;
    part def DriveIF_Def { port p : DriveIF; }
    part def ShaftPort_d_Def { port p : ShaftPort_d; }
    part def HandPort_Distractor_Def { port p : HandPort; }
    part def SubSystem_Context {
        part comp_a_343e : DriveIF_Def;
        part comp_b_463f : ShaftPort_d_Def;
        part comp_distractor_4a0a : Ha

In [53]:
truth = ds_test[i]["code_response"] # ground truth
print(truth)

CODE STATUS = INCORRECT 
FIX: 
```
package Vehicle_Remix_0939_418a {
    port def DriveIF;
    port def HandPort;
    port def ShaftPort_d;
    part def DriveIF_Def { port p : DriveIF; }
    part def ShaftPort_d_Def { port p : ShaftPort_d; }
    part def HandPort_Distractor_Def { port p : HandPort; }
    part def SubSystem_Context {
        part comp_a_343e : DriveIF_Def;
        part comp_b_463f : ShaftPort_d_Def;
        part comp_distractor_4a0a : HandPort_Distractor_Def;
        connect comp_a_343e.p to comp_b_463f.p;
    }
}
```
